# Document to Audio LangGraph

To keep this completely free while maintaining high quality, you can leverage generous free-tier APIs and open-source local models:

- **Orchestration:** LangGraph (Open-source, free)
- **LLM (Scriptwriter & Fact-Checker):** Gemini 1.5 Flash via Google AI Studio — incredibly generous free tier, natively processes massive documents (up to 1M+ tokens), highly capable of self-correction
- **Text-to-Speech (Audio):** Kokoro-82M or Edge-TTS — stellar, lightweight, open-source TTS model that runs beautifully on standard local CPU for free, producing studio-grade AI voices
- **Parsing:** pypdf or python-docx for document ingestion (Open-source, free)

## Workflow Architecture

Before jumping into the plan, here is how the data and control will flow through your LangGraph state machine.

### The Shared Graph State

Every node in LangGraph reads from and writes to a shared, typed state. For your project, the state will track:

- `document_text` — The raw text extracted from the upload
- `script` — The current draft of the podcast script
- `feedback` — The evaluation notes from the fact-checker
- `is_factual` — A boolean flag (True/False) indicating if it passed the check
- `iteration_count` — An integer tracking how many rewrites have occurred (capped at 3)
- `audio_path` — The file path to the final output audio file

## The Step-by-Step Project Plan

### Phase 1: Environment Setup & Document Parsing

**Goal:** Read the uploaded file and get your environment ready.

**Tasks:**
- Install dependencies: `pip install langgraph langchain-google-genai pypdf python-docx kokoro soundfile`
- Set up your free Google AI Studio API key
- Write a helper utility function using pypdf to extract text from an uploaded document and save it to the graph state

### Phase 2: Defining the Graph Nodes (The Agents)

**Goal:** Code the isolated Python functions that act as your workflow's "steps."

**Tasks:**
- **Scriptwriter Node** — Takes `document_text` (and any existing `feedback`), instructs Gemini to write/adjust the script, and updates the `script` state
- **Fact-Checker Node** — Instructs Gemini to compare the new `script` against the original `document_text`. It outputs a critique (`feedback`), flips `is_factual` to True if accurate, and increments `iteration_count`
- **Audio Generator Node** — Takes the finalized `script` and passes it to Kokoro TTS to compile the audio file

### Phase 3: Implementing Loop Control & Conditional Edges

**Goal:** Stitch the nodes together and enforce the "max 3 rewrites" rule.

**Tasks:**
- Create a conditional routing function (`should_continue`) that looks at `is_factual` and `iteration_count`
- Build and compile the StateGraph using the blueprint below

### Phase 4: Testing & Polish

**Goal:** Run the graph and test the boundaries.

**Tasks:**
- Feed the graph a document with a known tricky or ambiguous statement to force the Fact-Checker to catch it and trigger a rewrite loop
- Verify that the loop gracefully breaks and moves to TTS on the 3rd attempt, even if the script isn't 100% perfect, to avoid infinite API loops

## Current Flow Diagram

```mermaid
graph TD
    Start([START]) --> Parse[parse_document]

    Parse --> Chunk[chunk_document<br/>>16k chars -> 8-10k chunks<br/>else single-chunk array]

    Chunk --> LoopStart{Are there remaining chunks?}
    LoopStart -- No --> End([END])
    LoopStart -- Yes: take next chunk --> GenScript[generate_script]

    GenScript --> FactCheck[fact_check_script<br/>compare vs current chunk]
    FactCheck --> Decision{is_factual OR<br/>iteration_count >= 3?}
    Decision -- No --> GenScript
    Decision -- Yes --> GenAudio[generate_audio<br/>append chunk audio<br/>advance to next chunk]

    GenAudio --> LoopStart

    style Start fill:#13751f
    style Parse fill:#0a2f6d
    style Chunk fill:#0a2f6d
    style GenScript fill:#0a2f6d
    style FactCheck fill:#0a2f6d
    style Decision fill:#bab407
    style LoopStart fill:#bab407
    style GenAudio fill:#ba07b1
    style End fill:#f23c0e
```

**Two loops:** an *outer* loop over document chunks (`has_remaining_chunks`) and an
*inner* rewrite loop per chunk (`should_rewrite`) capped at 3 attempts. Each chunk's
audio is appended to a single combined `<document_name>_podcast.wav`.

In [ ]:
# Uncomment this to install all necessary dependencies.
# %pip install langgraph langchain-google-genai langchain-text-splitters pypdf python-docx kokoro soundfile

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from typing import TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# Document Parsing
from pypdf import PdfReader          # PDF
from docx import Document            # DOCX (python-docx package)

# TTS + audio output
from kokoro import KPipeline
import soundfile as sf
import numpy as np

# Load GOOGLE_API_KEY from .env into the environment.
# langchain-google-genai picks it up automatically; we never hardcode the key.
load_dotenv()

# Single shared LLM for both the scriptwriter and the fact-checker.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# --- Chunking configuration --------------------------------------------------
# Documents at or under the threshold are processed as a single chunk. Longer
# documents are split into ~CHUNK_SIZE-char pieces so each fits comfortably in
# one generate -> fact-check -> audio pass. Overlap is 0 to avoid duplicated
# audio across chunk boundaries.
CHUNK_THRESHOLD = 16000
CHUNK_SIZE = 9000
CHUNK_OVERLAP = 0

# Kokoro is expensive to construct, so build the pipeline once and reuse it
# across every generate_audio call (and across rewrite loops).
_tts_pipeline = None

def get_tts_pipeline():
    global _tts_pipeline
    if _tts_pipeline is None:
        _tts_pipeline = KPipeline(lang_code="a")  # 'a' = American English
    return _tts_pipeline


# --- Document parsing helper -------------------------------------------------
def parse_document(path: str) -> str:
    """Extract raw text from a .pdf or .docx file."""
    lower = path.lower()
    if lower.endswith(".pdf"):
        reader = PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    if lower.endswith(".docx"):
        doc = Document(path)
        return "\n".join(para.text for para in doc.paragraphs)
    raise ValueError(f"Unsupported file type: {path} (expected .pdf or .docx)")


# 1. Define the State
class PodcastState(TypedDict):
    document_path: str        # input: path to the source .pdf/.docx
    document_name: str        # input: base name used for the output audio file
    document_text: str        # raw text, populated by parse_document_node
    chunks: list[str]         # the document split into processable pieces
    current_chunk_index: int  # outer-loop pointer into `chunks`
    script: str               # current chunk's draft script
    feedback: str             # fact-checker critique for the current chunk
    is_factual: bool          # did the current chunk's script pass the check
    iteration_count: int      # per-chunk rewrite count (reset between chunks)
    audio_segments: list      # accumulated per-chunk audio arrays
    audio_path: str           # path to the final combined audio file


# Structured output schema for the fact-checker, so is_factual is reliable.
class FactCheck(BaseModel):
    is_factual: bool = Field(description="True if the script is faithful to the source document.")
    feedback: str = Field(description="Specific, actionable critique of any inaccuracies; brief confirmation if accurate.")


# 2. Define Node Functions
def parse_document_node(state: PodcastState):
    # Extract raw text from the uploaded document into the graph state.
    text = parse_document(state["document_path"])
    print(f"Parsed {len(text)} characters from {state['document_path']}")
    return {"document_text": text}


def chunk_document(state: PodcastState):
    # Short documents are a single chunk; long ones are split on natural
    # boundaries. This also initializes all outer-loop state.
    text = state["document_text"]
    if len(text) <= CHUNK_THRESHOLD:
        chunks = [text]
    else:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        chunks = splitter.split_text(text)

    print(f"Split into {len(chunks)} chunk(s)")
    return {
        "chunks": chunks,
        "current_chunk_index": 0,
        "iteration_count": 0,
        "feedback": "",
        "audio_segments": [],
    }


def generate_script(state: PodcastState):
    # Gemini writes/rewrites the script for the current chunk, using any prior feedback.
    idx = state["current_chunk_index"]
    chunk = state["chunks"][idx]
    print(
        f"Generating script for chunk {idx + 1}/{len(state['chunks'])} "
        f"(Iteration {state['iteration_count'] + 1})"
    )

    prompt = (
        "You are a podcast scriptwriter. Using ONLY the information in the source "
        "section below, write a single-narrator podcast script.\n\n"
        "Requirements:\n"
        "- This is ONE CONTINUOUS SECTION of a larger document. Do NOT add any "
        "intro, greeting, outro, sign-off, or transition phrases; produce only the "
        "spoken-word coverage of this section's content.\n"
        "- Plain spoken-word prose only. No stage directions, no speaker labels, "
        "no markdown, no headings, no bullet points.\n"
        "- Stay strictly faithful to the source; do not invent facts.\n"
        "- Where possible, quote the source directly.\n\n"
        f"SOURCE SECTION:\n{chunk}\n"
    )

    feedback = state.get("feedback")
    if feedback:
        prompt += (
            "\nA fact-checker reviewed your previous draft and found issues. "
            "Revise the script to address this critique while keeping it faithful "
            f"to the source:\n{feedback}\n"
        )

    return {"script": llm.invoke(prompt).content}


def fact_check_script(state: PodcastState):
    # Gemini compares the script against the current chunk and returns structured output.
    print("Checking for hallucinations...")

    chunk = state["chunks"][state["current_chunk_index"]]
    checker = llm.with_structured_output(FactCheck)
    prompt = (
        "You are a strict fact-checker. Compare the PODCAST SCRIPT against the "
        "SOURCE SECTION.\n"
        "- If the script adds claims absent from the source, or contradicts it, set "
        "is_factual=False and give specific, actionable feedback naming what to fix.\n"
        "- If the script is faithful to the source, set is_factual=True with a brief "
        "confirmation.\n\n"
        f"SOURCE SECTION:\n{chunk}\n\n"
        f"PODCAST SCRIPT:\n{state['script']}\n"
    )

    result = checker.invoke(prompt)
    return {
        "feedback": result.feedback,
        "is_factual": result.is_factual,
        "iteration_count": state["iteration_count"] + 1,
    }


def generate_audio(state: PodcastState):
    # Synthesize audio for the current chunk's finalized script, append it to the
    # running output, and advance the outer loop to the next chunk.
    idx = state["current_chunk_index"]
    print(f"Generating audio for chunk {idx + 1}/{len(state['chunks'])}...")

    pipeline = get_tts_pipeline()
    segment = np.concatenate(
        [audio for _graphemes, _phonemes, audio in pipeline(state["script"], voice="af_heart")]
    )

    audio_segments = state["audio_segments"] + [segment]

    # Write the running concatenation each pass, so the file is complete whenever
    # the outer loop ends (no separate finalize node, matching the flow diagram).
    audio_path = f"{state['document_name']}_podcast.wav"
    sf.write(audio_path, np.concatenate(audio_segments), 24000)  # Kokoro emits 24 kHz audio

    return {
        "audio_segments": audio_segments,
        "audio_path": audio_path,
        "current_chunk_index": idx + 1,
        "iteration_count": 0,  # reset the rewrite loop for the next chunk
        "feedback": "",        # don't leak this chunk's critique into the next
    }


# 3. Conditional Router Functions
def should_rewrite(state: PodcastState):
    # Inner loop: stop after a pass OR after 3 rewrites (no infinite loops).
    # The cap-3 guard is the safety boundary of the graph; enforced per chunk.
    if state["is_factual"] or state["iteration_count"] >= 3:
        return "generate_audio"
    return "generate_script"


def has_remaining_chunks(state: PodcastState):
    # Outer loop: process the next chunk, or finish when all are done.
    if state["current_chunk_index"] < len(state["chunks"]):
        return "generate_script"
    return END


# 4. Build the Graph
workflow = StateGraph(PodcastState)

# Add Nodes
workflow.add_node("parse_document", parse_document_node)
workflow.add_node("chunk_document", chunk_document)
workflow.add_node("generate_script", generate_script)
workflow.add_node("fact_check_script", fact_check_script)
workflow.add_node("generate_audio", generate_audio)

# Construct Edges
workflow.add_edge(START, "parse_document")
workflow.add_edge("parse_document", "chunk_document")

# Outer loop entry: start the first chunk (or finish immediately if none).
workflow.add_conditional_edges(
    "chunk_document",
    has_remaining_chunks,
    {"generate_script": "generate_script", END: END},
)

workflow.add_edge("generate_script", "fact_check_script")

# Inner loop: rewrite the chunk or accept it and move to audio.
workflow.add_conditional_edges(
    "fact_check_script",
    should_rewrite,
    {"generate_script": "generate_script", "generate_audio": "generate_audio"},
)

# Outer loop continuation: next chunk or END.
workflow.add_conditional_edges(
    "generate_audio",
    has_remaining_chunks,
    {"generate_script": "generate_script", END: END},
)

# Compile the Graph
app = workflow.compile()

c:\Users\Zeeshan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# --- Run the pipeline --------------------------------------------------------
# Point this at your source document (.pdf or .docx). Parsing and chunking now
# happen inside the graph, so we only pass the path and output name.
DOCUMENT_PATH = r"C:\Users\Zeeshan\Documents\Module_10_Machine_Learning_Study_Guide.docx"
DOCUMENT_NAME = "Module_10_Machine_Learning_Study_Guide"

# Each chunk takes several super-steps (generate -> fact-check -> [rewrites] ->
# audio), so the default recursion_limit of 25 can be hit on long documents.
# Give the graph generous headroom for many chunks.
result = app.invoke(
    {"document_path": DOCUMENT_PATH, "document_name": DOCUMENT_NAME},
    config={"recursion_limit": 200},
)

print("\nDone.")
print("Chunks processed:", len(result["chunks"]))
print("Final chunk passed fact-check:", result["is_factual"], "| rewrites on last chunk:", result["iteration_count"])
print("Audio written to:", result["audio_path"])

Parsed 43746 characters from C:\Users\Zeeshan\Documents\Module_10_Machine_Learning_Study_Guide.docx
Split into 7 chunk(s)
Generating script for chunk 1/7 (Iteration 1)
Checking for hallucinations...
Generating audio for chunk 1/7...


c:\Users\Zeeshan\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:1009: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
c:\Users\Zeeshan\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Generating script for chunk 2/7 (Iteration 1)
Checking for hallucinations...
Generating audio for chunk 2/7...
Generating script for chunk 3/7 (Iteration 1)
Checking for hallucinations...
Generating audio for chunk 3/7...
Generating script for chunk 4/7 (Iteration 1)
Checking for hallucinations...
Generating audio for chunk 4/7...
Generating script for chunk 5/7 (Iteration 1)
Checking for hallucinations...
Generating audio for chunk 5/7...
Generating script for chunk 6/7 (Iteration 1)
Checking for hallucinations...
Generating audio for chunk 6/7...
Generating script for chunk 7/7 (Iteration 1)
Checking for hallucinations...
Generating audio for chunk 7/7...

Done.
Chunks processed: 7
Final chunk passed fact-check: True | rewrites on last chunk: 0
Audio written to: Module_10_Machine_Learning_Study_Guide_podcast.wav
